# Plan B Cloud Execution Pipeline: Genome-Wide 83-Locus SuSiE Colocalization, Independent sn-eQTL Replication, BrainMeta sQTL SMR, and External snRNA-seq Validation

This notebook implements all four external-compute upgrades recommended for *Molecular Psychiatry* submission:
1. **Part A (Systematic 83-Locus Multi-Causal Fine-Mapping)**: Runs `susieR::susie_rss`, `susieR::kriging_rss` LD-mismatch diagnostics, and `coloc::coloc.susie` across all 83 genome-wide MDD-ANX pleiotropy windows (`aim2_83_coloc_leads_annotated.tsv`) in both full EUR MDD (`MDD_EUR`) and UKBB-excluded MDD (`MDD_noUKBB`) vs. ANX (`ANX_EUR`).
2. **Part B (Independent Cell-Type *cis*-eQTL Colocalization)**: Replicates 20q13.12 (`SLC12A5` and positional neighbors) and harmonized allele polarity in **PsychENCODE brainSCOPE** (`Emani et al. 2024, Science`, *N* = 388, open-access scQTL portal) and **Fujita et al. 2024** (*N* = 424, Synapse `syn52335732` via optional DUC token).
3. **Part C (BrainMeta v2 Isoform-Level sQTL SMR)**: Tests *SLC12A5* splicing QTLs (`KCC2a`/`KCC2b`/`EXON6B` proxy introns) using the Yang Lab BrainMeta v2 `cis-sQTL` BESD archive (`Qi et al. 2022, Nat. Genet.`).
4. **Part D (Independent Postmortem DLPFC snRNA-seq Replication)**: Uniform donor-level `limma-voom` (+ `HSPA1A` & `sva` sensitivity) template for an independent MDD cortical snRNA-seq `h5ad` cohort.

**Recommended Colab Runtime**: `CPU + High-RAM (51 GB)` with Google Drive mounted for resumable caching.


In [ ]:
# [1] Mount Google Drive, Check Resources, & Install System / R / Python Dependencies
import os, shutil, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    candidates = [
        '/content/drive/MyDrive/MDD_New_Project_Pipelines_2026/MDD_PlanB_Replication',
        '/content/drive/MyDrive/MDD_PlanB_Replication',
        '/content/MDD_PlanB_Replication',
    ]
    DRIVE_ROOT = next((d for d in candidates if os.path.exists(d)), '/content/drive/MyDrive/MDD_New_Project_Pipelines_2026/MDD_PlanB_Replication')
except Exception:
    DRIVE_ROOT = '/content/MDD_PlanB_Replication'

CACHE_DIR = os.path.join(DRIVE_ROOT, 'cache_downloads')
OUT_DIR = os.path.join(DRIVE_ROOT, 'outputs')
WORK_DIR = '/content/planb_nvme'
for d in [CACHE_DIR, OUT_DIR, WORK_DIR, os.path.join(WORK_DIR, 'ld'), os.path.join(WORK_DIR, 'gwas'), os.path.join(WORK_DIR, 'smr')]:
    os.makedirs(d, exist_ok=True)

print(f'[Setup] DRIVE_ROOT={DRIVE_ROOT}, Free NVMe={shutil.disk_usage("/content").free / 1e9:.1f} GB')

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    'plink1.9', 'unzip', 'wget', 'curl', 'git', 'cmake', 'g++',
    'libopenblas-dev', 'libgsl-dev', 'r-base', 'r-cran-data.table',
    'r-cran-remotes', 'r-cran-matrix', 'r-cran-rcpp', 'r-cran-biocmanager'
], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'bottleneck>=1.5.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'scipy', 'requests', 'synapseclient', 'anndata', 'scanpy'], check=True)

r_install = r'''
options(repos = c(CRAN = "https://cloud.r-project.org"))
for (p in c("data.table", "coloc", "susieR")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
}
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
for (bp in c("edgeR", "limma", "sva")) {
  if (!requireNamespace(bp, quietly = TRUE)) BiocManager::install(bp, ask = FALSE, update = FALSE)
}
cat("R packages ready: coloc=", as.character(packageVersion("coloc")),
    "susieR=", as.character(packageVersion("susieR")), "\n")
'''
subprocess.run(['Rscript', '-e', r_install], check=True)
print('[OK] Cell [1] environment setup complete.')


In [ ]:
# [2] Stage Yang Lab SMR Binary & 1000 Genomes Phase 3 EUR LD Reference Panel
import urllib.request, zipfile

smr_bin = '/usr/local/bin/smr'
if not os.path.exists(smr_bin):
    smr_zip = os.path.join(CACHE_DIR, 'smr_linux.zip')
    if not os.path.exists(smr_zip):
        for url in [
            'https://yanglab.westlake.edu.cn/software/smr/download/smr-1.4.3-linux-x86_64.zip',
            'https://yanglab.westlake.edu.cn/software/smr/download/smr-1.3.1-linux-x86_64.zip',
        ]:
            subprocess.run(['wget', '-q', '--timeout=30', '--user-agent=Mozilla/5.0', url, '-O', smr_zip], check=False)
            if os.path.exists(smr_zip) and zipfile.is_zipfile(smr_zip):
                break
    if os.path.exists(smr_zip) and zipfile.is_zipfile(smr_zip):
        subprocess.run(['unzip', '-o', '-j', smr_zip, '*smr*', '-d', '/tmp/smr_ext'], check=False)
        for root, _, files in os.walk('/tmp/smr_ext'):
            for f in files:
                if f.startswith('smr') and not f.endswith('.zip'):
                    src = os.path.join(root, f)
                    shutil.copy2(src, smr_bin)
                    os.chmod(smr_bin, 0o755)
    print('[SMR] Available:', os.path.exists(smr_bin))

ld_cache = os.path.join(CACHE_DIR, '1000G_EUR')
os.makedirs(ld_cache, exist_ok=True)
g1k_zip = os.path.join(ld_cache, 'g1000_eur.zip')
if not os.path.exists(os.path.join(ld_cache, 'g1000_eur.bed')):
    if not os.path.exists(g1k_zip):
        surf_url = 'https://vu.data.surfsara.nl/index.php/s/VZNByNwpD8qqINe/download'
        subprocess.run(['wget', '-c', '--user-agent=Mozilla/5.0', surf_url, '-O', g1k_zip], check=True)
    subprocess.run(['unzip', '-o', g1k_zip, '-d', ld_cache], check=True)

for ext in ['bed', 'bim', 'fam']:
    shutil.copy2(os.path.join(ld_cache, f'g1000_eur.{ext}'), os.path.join(WORK_DIR, 'ld', f'g1000_eur.{ext}'))
print('[OK] Cell [2] SMR binary & 1000G EUR LD panel staged on NVMe.')


In [ ]:
# [3] Download Open-Access PGC MDD 2025 (Full + noUKBB) & Friligkou 2024 ANX EUR GWAS
import json, requests

def fetch_figshare_article(article_id: str, dest_dir: str, patterns: list[str]) -> list[str]:
    r = requests.get(f'https://api.figshare.com/v2/articles/{article_id}', timeout=60)
    r.raise_for_status()
    saved = []
    for f in r.json().get('files', []):
        name = f['name']
        if any(p in name for p in patterns):
            out = os.path.join(dest_dir, name)
            if not os.path.exists(out):
                print(f'  Downloading {name} ({f["size"]/1e6:.1f} MB)...')
                subprocess.run(['wget', '-c', '-q', f['download_url'], '-O', out], check=True)
            saved.append(out)
    return saved

def fetch_zenodo_record(record_id: str, dest_dir: str, patterns: list[str]) -> list[str]:
    r = requests.get(f'https://zenodo.org/api/records/{record_id}', timeout=60)
    r.raise_for_status()
    saved = []
    for f in r.json().get('files', []):
        key = f['key']
        if any(p in key for p in patterns):
            out = os.path.join(dest_dir, key)
            if not os.path.exists(out):
                print(f'  Downloading {key} ({f["size"]/1e6:.1f} MB)...')
                subprocess.run(['wget', '-c', '-q', f['links']['self'], '-O', out], check=True)
            saved.append(out)
    return saved

gwas_cache = os.path.join(CACHE_DIR, 'gwas')
os.makedirs(gwas_cache, exist_ok=True)
mdd_files = fetch_figshare_article('27061255', gwas_cache, ['eur', 'noUKBB', 'no23andMe'])
anx_files = fetch_zenodo_record('13135834', gwas_cache, ['EUR', 'eur', 'ANX', 'anxiety'])
print('[GWAS Files]', {'MDD': [os.path.basename(x) for x in mdd_files], 'ANX': [os.path.basename(x) for x in anx_files]})


## Part A: Genome-Wide 83-Locus `kriging_rss` + `susie_rss` + `coloc.susie` (`MDD_EUR` & `MDD_noUKBB` vs. `ANX_EUR`)

Upload `aim2_83_coloc_leads_annotated.tsv` (from `03_Aim2_Aim3_Server_Scripts/outputs/tables/`) to `/content/drive/MyDrive/MDD_PlanB_Replication/` or keep a local path; the cell below builds ±500 kb windows around all 83 lead SNPs, filters LD-mismatched SNPs via `kriging_rss` (`log10LR > 2` and `|z| > 2`), runs multi-causal `susie_rss(L=10)` for `MDD_EUR`, `MDD_noUKBB`, and `ANX_EUR`, and tabulates `coloc.susie` results across all 83 loci.


In [ ]:
# [4] Part A: Run Systematic 83-Locus SuSiE + kriging_rss + coloc.susie
import glob, os, pandas as pd, numpy as np

leads_candidates = [
    os.path.join(DRIVE_ROOT, 'aim2_83_coloc_leads_annotated.tsv'),
    '/content/drive/MyDrive/MDD_New_Project_Pipelines_2026/MDD_PlanB_Replication/aim2_83_coloc_leads_annotated.tsv',
    '/content/drive/MyDrive/MDD_PlanB_Replication/aim2_83_coloc_leads_annotated.tsv',
    '/content/aim2_83_coloc_leads_annotated.tsv',
]
if os.path.exists('/content/drive/MyDrive'):
    leads_candidates.extend(glob.glob('/content/drive/MyDrive/**/aim2_83_coloc_leads_annotated.tsv', recursive=True))

leads_path = next((p for p in leads_candidates if os.path.exists(p)), None)
assert leads_path is not None, (
    f'Upload aim2_83_coloc_leads_annotated.tsv to {DRIVE_ROOT} or /content/drive/MyDrive/MDD_New_Project_Pipelines_2026/MDD_PlanB_Replication'
)
leads = pd.read_csv(leads_path, sep='\t')
print(f'[Part A] Loaded {len(leads)} genome-wide MDD-ANX pleiotropy lead SNPs from {leads_path}.')

# Inspect header of downloaded MDD & ANX sumstats to harmonize columns dynamically
def read_sumstats_flexible(path: str) -> pd.DataFrame:
    import gzip
    opener = gzip.open if path.endswith('.gz') else open
    skip = 0
    first_col_line = None
    with opener(path, 'rt', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('##'):
                skip += 1
            else:
                first_col_line = line
                break
    if first_col_line is None:
        raise ValueError(f'Empty file: {path}')
    sep = '\t' if '\t' in first_col_line else r'\s+'
    df = pd.read_csv(path, sep=sep, skiprows=skip, compression='infer', low_memory=False)
    df.columns = [str(c).lstrip('#').strip() for c in df.columns]
    cmap = {c.lower(): c for c in df.columns}
    ren = {}
    for target, opts in [
        ('SNP', ['snp', 'id', 'rsid', 'markername', 'rs']),
        ('CHR', ['chr', 'chrom', 'chromosome']),
        ('BP', ['bp', 'pos', 'position', 'base_pair_location']),
        ('A1', ['a1', 'ea', 'effect_allele', 'alt']),
        ('A2', ['a2', 'nea', 'other_allele', 'ref']),
        ('BETA', ['beta', 'b', 'effect', 'log_or']),
        ('SE', ['se', 'standard_error', 'se_gc']),
        ('P', ['p', 'pval', 'p_value', 'p.value']),
        ('N', ['n', 'neff', 'n_eff', 'totaln', 'n_total']),
    ]:
        for o in opts:
            if o in cmap:
                ren[cmap[o]] = target
                break
    df = df.rename(columns=ren)
    if 'BETA' not in df.columns and 'OR' in df.columns:
        df['BETA'] = np.log(df['OR'])
    if 'Z' not in df.columns and {'BETA', 'SE'}.issubset(df.columns):
        df['Z'] = df['BETA'] / df['SE']
    if 'N' not in df.columns:
        if {'NCAS', 'NCON'}.issubset(df.columns):
            df['N'] = 4.0 / (1.0 / df['NCAS'] + 1.0 / df['NCON'])
        else:
            df['N'] = 1096458.0
    return df[['SNP', 'CHR', 'BP', 'A1', 'A2', 'BETA', 'SE', 'Z', 'P', 'N']].dropna()

# Select main EUR MDD and ANX files (filter out tiny metadata .txt files)
mdd_large_files = [p for p in mdd_files if not p.endswith('.txt')]
mdd_fmt_files = [p for p in mdd_large_files if '_formatted.tsv.gz' in p] or mdd_large_files
mdd_main_p = next((p for p in mdd_fmt_files if 'no23andMe_eur' in os.path.basename(p) and 'noUKBB' not in os.path.basename(p)), mdd_fmt_files[0])
mdd_nouk_p = next((p for p in mdd_fmt_files if 'noUKBB' in os.path.basename(p)), None)

anx_large_files = [p for p in anx_files if not p.endswith('.txt')] or anx_files
anx_main_p = next((p for p in anx_large_files if 'EUR' in os.path.basename(p) or 'eur' in os.path.basename(p)), anx_large_files[0])

print(f'  Loading MDD_EUR: {os.path.basename(mdd_main_p)}')
df_mdd = read_sumstats_flexible(mdd_main_p)
print(f'  Loading ANX_EUR: {os.path.basename(anx_main_p)}')
df_anx = read_sumstats_flexible(anx_main_p)
df_nouk = read_sumstats_flexible(mdd_nouk_p) if mdd_nouk_p else None

bim = pd.read_csv(os.path.join(WORK_DIR, 'ld', 'g1000_eur.bim'), sep='\t', header=None, names=['CHR', 'SNP', 'CM', 'BP', 'REF_A1', 'REF_A2'])
bim_set = set(bim['SNP'])

loci83_dir = os.path.join(WORK_DIR, 'loci83')
os.makedirs(loci83_dir, exist_ok=True)
manifest = []
for idx, row in leads.iterrows():
    chr_i, bp_i, snp_i = int(row['CHR']), int(row['BP']), str(row['SNP'])
    w0, w1 = max(1, bp_i - 500_000), bp_i + 500_000
    tag = f'L{idx+1:02d}_chr{chr_i}_{snp_i}'
    sub_m = df_mdd[(df_mdd['CHR'] == chr_i) & (df_mdd['BP'].between(w0, w1))]
    sub_a = df_anx[(df_anx['CHR'] == chr_i) & (df_anx['BP'].between(w0, w1))]
    merged = sub_m.merge(sub_a, on='SNP', suffixes=('_MDD', '_ANX'))
    merged = merged[merged['SNP'].isin(bim_set)].drop_duplicates(subset=['SNP']).sort_values('BP_MDD')
    if len(merged) < 50:
        continue
    # Align ANX alleles to MDD alleles
    same_dir = (merged['A1_MDD'] == merged['A1_ANX']) & (merged['A2_MDD'] == merged['A2_ANX'])
    flip_dir = (merged['A1_MDD'] == merged['A2_ANX']) & (merged['A2_MDD'] == merged['A1_ANX'])
    merged = merged[same_dir | flip_dir].copy()
    flip_mask = (merged['A1_MDD'] == merged['A2_ANX'])
    merged.loc[flip_mask, 'BETA_ANX'] *= -1.0
    merged.loc[flip_mask, 'Z_ANX'] *= -1.0
    if df_nouk is not None:
        sub_nk = df_nouk[(df_nouk['CHR'] == chr_i) & (df_nouk['BP'].between(w0, w1))][['SNP', 'A1', 'A2', 'BETA', 'SE', 'Z', 'P', 'N']]
        merged = merged.merge(sub_nk, on='SNP', how='left', suffixes=('', '_NOUK'))
        nk_flip = (merged['A1_MDD'] == merged['A2'])
        merged.loc[nk_flip, 'BETA'] *= -1.0
        merged.loc[nk_flip, 'Z'] *= -1.0
    out_tsv = os.path.join(loci83_dir, f'{tag}.tsv')
    merged.to_csv(out_tsv, sep='\t', index=False)
    manifest.append({'tag': tag, 'chr': chr_i, 'bp_start': int(merged['BP_MDD'].min()), 'bp_end': int(merged['BP_MDD'].max()), 'nsnps': len(merged), 'lead_snp': snp_i, 'file': out_tsv})

pd.DataFrame(manifest).to_csv(os.path.join(loci83_dir, 'manifest_83loci.tsv'), sep='\t', index=False)
print(f'[Part A] Prepared {len(manifest)} regional harmonized tables. Running PLINK LD + R kriging_rss + coloc.susie...')

r_part_a = fr'''
suppressPackageStartupMessages({{ library(data.table); library(susieR); library(coloc) }})
man <- fread("{loci83_dir}/manifest_83loci.tsv")
ld_bfile <- "{WORK_DIR}/ld/g1000_eur"
all_res <- list()

for (k in seq_len(nrow(man))) {{
  m <- man[k]
  dt <- fread(m$file)
  snp_f <- file.path("{loci83_dir}", paste0(m$tag, ".snps"))
  fwrite(dt[, .(SNP)], snp_f, col.names = FALSE)
  pref <- file.path("{loci83_dir}", m$tag)
  cmd <- sprintf("plink1.9 --bfile %s --extract %s --r square --write-snplist --out %s > /dev/null 2>&1", ld_bfile, snp_f, pref)
  system(cmd)
  if (!file.exists(paste0(pref, ".ld"))) next
  R <- as.matrix(fread(paste0(pref, ".ld")))
  snplist <- fread(paste0(pref, ".snplist"), header = FALSE)$V1
  dt <- dt[match(snplist, dt$SNP)]
  dt <- dt[!is.na(Z_MDD) & !is.na(Z_ANX)]
  rownames(R) <- colnames(R) <- snplist

  # LD mismatch diagnostic via kriging_rss
  cond_m <- tryCatch(kriging_rss(dt$Z_MDD, R, n = median(dt$N_MDD)), error = function(e) NULL)
  keep <- rep(TRUE, nrow(dt))
  n_mismatch <- 0L
  if (!is.null(cond_m) && "conditional_dist" %in% names(cond_m)) {{
    cd <- cond_m$conditional_dist
    bad <- which(cd$logLR > 2 & abs(cd$z) > 2)
    if (length(bad) > 0 && length(bad) < nrow(dt) * 0.2) {{
      keep[bad] <- FALSE
      n_mismatch <- length(bad)
    }}
  }}
  dt_k <- dt[keep]
  R_k <- R[keep, keep, drop = FALSE]
  if (nrow(dt_k) < 30) next

  d_mdd <- list(beta = dt_k$BETA_MDD, varbeta = dt_k$SE_MDD^2, snp = dt_k$SNP, position = dt_k$BP_MDD, type = "cc", N = median(dt_k$N_MDD), LD = R_k)
  d_anx <- list(beta = dt_k$BETA_ANX, varbeta = dt_k$SE_ANX^2, snp = dt_k$SNP, position = dt_k$BP_ANX, type = "cc", N = median(dt_k$N_ANX), LD = R_k)

  s_mdd <- tryCatch(runsusie(d_mdd, L = 10), error = function(e) NULL)
  s_anx <- tryCatch(runsusie(d_anx, L = 10), error = function(e) NULL)
  abf <- coloc.abf(d_mdd, d_anx)$summary

  # Also run UKBB-excluded MDD_noUKBB sensitivity if non-missing
  nouk_h4 <- NA_real_
  if ("BETA" %in% names(dt_k) && sum(!is.na(dt_k$BETA)) > 30) {{
    keep_nk <- !is.na(dt_k$BETA) & !is.na(dt_k$SE) & dt_k$SE > 0
    if (sum(keep_nk) >= 30) {{
      d_nk <- list(beta = dt_k$BETA[keep_nk], varbeta = dt_k$SE[keep_nk]^2, snp = dt_k$SNP[keep_nk],
                   position = dt_k$BP_MDD[keep_nk], type = "cc", N = median(dt_k$N[keep_nk]),
                   LD = R_k[keep_nk, keep_nk, drop = FALSE])
      d_a_nk <- list(beta = dt_k$BETA_ANX[keep_nk], varbeta = dt_k$SE_ANX[keep_nk]^2, snp = dt_k$SNP[keep_nk],
                     position = dt_k$BP_ANX[keep_nk], type = "cc", N = median(dt_k$N_ANX[keep_nk]),
                     LD = R_k[keep_nk, keep_nk, drop = FALSE])
      s_nk <- tryCatch(runsusie(d_nk, L = 10), error = function(e) NULL)
      s_a_nk <- tryCatch(runsusie(d_a_nk, L = 10), error = function(e) NULL)
      if (!is.null(s_nk) && !is.null(s_a_nk) && length(s_nk$sets$cs) > 0 && length(s_a_nk$sets$cs) > 0) {{
        nouk_h4 <- max(coloc.susie(s_nk, s_a_nk)$summary$PP.H4.abf, na.rm = TRUE)
      }} else {{
        nouk_h4 <- as.numeric(coloc.abf(d_nk, d_a_nk)$summary["PP.H4.abf"])
      }}
    }}
  }}

  if (!is.null(s_mdd) && !is.null(s_anx) && length(s_mdd$sets$cs) > 0 && length(s_anx$sets$cs) > 0) {{
    cs <- coloc.susie(s_mdd, s_anx)
    sm <- as.data.table(cs$summary)
    sm[, `:=`(locus = m$tag, lead_snp = m$lead_snp, nsnps_clean = nrow(dt_k), kriging_dropped = n_mismatch,
              n_cs_mdd = length(s_mdd$sets$cs), n_cs_anx = length(s_anx$sets$cs),
              abf_PP_H4 = as.numeric(abf["PP.H4.abf"]), noUKBB_PP_H4 = nouk_h4)]
    all_res[[length(all_res) + 1]] <- sm
  }} else {{
    all_res[[length(all_res) + 1]] <- data.table(
      locus = m$tag, lead_snp = m$lead_snp, nsnps_clean = nrow(dt_k), kriging_dropped = n_mismatch,
      n_cs_mdd = ifelse(is.null(s_mdd), NA_integer_, length(s_mdd$sets$cs)),
      n_cs_anx = ifelse(is.null(s_anx), NA_integer_, length(s_anx$sets$cs)),
      PP.H3.abf = as.numeric(abf["PP.H3.abf"]), PP.H4.abf = as.numeric(abf["PP.H4.abf"]),
      abf_PP_H4 = as.numeric(abf["PP.H4.abf"]), noUKBB_PP_H4 = nouk_h4
    )
  }}
  if (k %% 10 == 0) cat(sprintf("  Processed %d / %d loci...\n", k, nrow(man)))
}}
out_dt <- rbindlist(all_res, fill = TRUE)
fwrite(out_dt, "{OUT_DIR}/planB_all83_loci_coloc_susie.tsv", sep = "\t")
cat("[SUCCESS] Saved 83-locus coloc.susie results:", nrow(out_dt), "rows\n")
'''
subprocess.run(['Rscript', '-e', r_part_a], check=True)


## Part B: Independent Single-Nucleus eQTL Replication (PsychENCODE brainSCOPE *N* = 388 & Fujita et al. *N* = 424)

1. **PsychENCODE brainSCOPE (`Emani et al. 2024, Science`)**: Open-access cell-type scQTL callsets from `brainscope.psychencode.org` across 28 cortical cell subclasses without DUC restrictions. Download the cell-type `cis`-eQTL summary archive into `CACHE_DIR/brainscope/` (auto-prompted if portal CDN layout shifts) and run exact `coloc.abf` across 20q13.12 (*SLC12A5* + 29 neighbors).
2. **Fujita et al. (*Nat. Genet.* 2024, *N* = 424)**: Automated `synapseclient` hook for `syn52335732` when `SYNAPSE_AUTH_TOKEN` is exported by an AD Knowledge Portal DUC-approved investigator.


In [ ]:
# [5] Part B: Independent Cell-Type cis-eQTL Colocalization at 20q13.12 (brainSCOPE + Optional Fujita syn52335732)
import glob

bs_dir = os.path.join(CACHE_DIR, 'brainscope')
os.makedirs(bs_dir, exist_ok=True)
# If user stages brainSCOPE PEC2 scQTL tarball/TSVs into CACHE_DIR/brainscope, extract automatically
for tb in glob.glob(os.path.join(bs_dir, '*.tar.gz')) + glob.glob(os.path.join(bs_dir, '*.zip')):
    if tb.endswith('.tar.gz'):
        subprocess.run(['tar', '-xzf', tb, '-C', bs_dir], check=False)
    else:
        subprocess.run(['unzip', '-n', tb, '-d', bs_dir], check=False)

# Optional Fujita et al. 2024 Synapse hook (requires DUC-approved personal access token)
fuj_dir = os.path.join(CACHE_DIR, 'fujita_syn52335732')
os.makedirs(fuj_dir, exist_ok=True)
syn_token = os.environ.get('SYNAPSE_AUTH_TOKEN', '').strip()
if syn_token:
    import synapseclient, synapseutils
    syn = synapseclient.Synapse()
    syn.login(authToken=syn_token, silent=True)
    synapseutils.syncFromSynapse(syn, entity='syn52335732', path=fuj_dir)
    print('[Fujita] Synced syn52335732 to', fuj_dir)
else:
    print('[Info] SYNAPSE_AUTH_TOKEN not set -> skipping Synapse syn52335732 download; running open-access brainSCOPE replication if files staged in:', bs_dir)

r_part_b = fr'''
suppressPackageStartupMessages({{ library(data.table); library(coloc) }})
eqtl_files <- c(
  list.files("{bs_dir}", pattern = ".*(eQTL|scQTL|cis).*\\.(tsv|txt|csv)(\\.gz)?$", full.names = TRUE, recursive = TRUE),
  list.files("{fuj_dir}", pattern = ".*(eQTL|cis).*\\.(tsv|txt|csv)(\\.gz)?$", full.names = TRUE, recursive = TRUE)
)
cat(sprintf("[Part B] Discovered %d external cell-type eQTL files.\n", length(eqtl_files)))
if (length(eqtl_files) == 0) {{
  cat("[Action Needed] Download open-access PEC2 cell-type eQTL files from http://brainscope.psychencode.org/ into:", "{bs_dir}", "\n")
}} else {{
  gwas_20 <- fread(list.files("{WORK_DIR}/loci83", pattern = "chr20_rs9074\\.tsv$", full.names = TRUE)[1])
  res_list <- list()
  for (ef in eqtl_files) {{
    eq <- tryCatch(fread(ef), error = function(e) NULL)
    if (is.null(eq) || nrow(eq) == 0) next
    # Normalize common SNP / P / BETA / GENE columns across PEC2 & Fujita schemas
    nm <- tolower(names(eq))
    names(eq) <- nm
    gcol <- intersect(c("gene", "gene_name", "symbol", "phenotype_id"), nm)[1]
    scol <- intersect(c("snp", "variant_id", "rsid"), nm)[1]
    pcol <- intersect(c("pval_nominal", "p", "pvalue", "p_val"), nm)[1]
    bcol <- intersect(c("slope", "beta", "b"), nm)[1]
    if (any(is.na(c(gcol, scol, pcol, bcol)))) next
    sub_e <- eq[get(scol) %in% gwas_20$SNP & get(gcol) %in% c("SLC12A5", "CD40", "STK4", "PLTP", "SLC2A10")]
    if (nrow(sub_e) < 10) next
    for (tg in unique(sub_e[[gcol]])) {{
      ge <- sub_e[get(gcol) == tg]
      mg <- merge(gwas_20, ge, by.x = "SNP", by.y = scol)
      if (nrow(mg) < 20) next
      n_e <- ifelse(grepl("fujita", tolower(ef)), 424, 388)
      ca <- coloc.abf(
        dataset1 = list(beta = mg$BETA_MDD, varbeta = mg$SE_MDD^2, snp = mg$SNP, type = "cc", N = median(mg$N_MDD)),
        dataset2 = list(pvalues = pmax(mg[[pcol]], 1e-300), snp = mg$SNP, type = "quant", N = n_e, MAF = 0.25)
      )$summary
      res_list[[length(res_list) + 1]] <- data.table(
        source = basename(ef), gene = tg, nsnps = nrow(mg),
        PP.H4_MDD = as.numeric(ca["PP.H4.abf"]), PP.H3_MDD = as.numeric(ca["PP.H3.abf"]),
        top_eqtl_p = min(mg[[pcol]])
      )
    }}
  }}
  if (length(res_list) > 0) {{
    fwrite(rbindlist(res_list), "{OUT_DIR}/planB_external_snQTL_coloc_20q13.tsv", sep = "\t")
    cat("[SUCCESS] External snQTL 20q13 colocalization saved.\n")
  }}
}}
'''
subprocess.run(['Rscript', '-e', r_part_b], check=True)


## Part C: Isoform-Level BrainMeta v2 *cis*-sQTL SMR + HEIDI (`SLC12A5` Splicing Check)

Downloads the public **BrainMeta v2 *cis*-sQTL BESD archive** (`Qi et al. 2022, Nat. Genet.`, ~9.0 GB tar.gz, chromosome 20 subset extracted directly) and runs `--smr-multi` against both `MDD_EUR` and `ANX_EUR` in `--chr 20` around `SLC12A5` (`chr20:44.15–45.20 Mb`) to resolve canonical `KCC2a`/`KCC2b` vs. `EXON6B` splicing QTLs.


In [ ]:
# [6] Part C: Run BrainMeta v2 cis-sQTL SMR + HEIDI on Chromosome 20 (SLC12A5)
sqtl_cache = os.path.join(CACHE_DIR, 'brainmeta_sqtl')
os.makedirs(sqtl_cache, exist_ok=True)
sqtl_tar = os.path.join(sqtl_cache, 'BrainMeta_cis_sqtl_summary.tar.gz')
sqtl_url = 'https://yanglab.westlake.edu.cn/software/smr/download/data/BrainMeta_cis_sqtl_summary.tar.gz'

# Build COJO .ma files from our chr20 harmonized table for fast targeted SMR
chr20_hits = sorted(glob.glob(os.path.join(WORK_DIR, 'loci83', '*chr20_rs9074.tsv')))
chr20_tsv = chr20_hits[0] if chr20_hits else ''
if chr20_tsv and os.path.exists(chr20_tsv):
    d20 = pd.read_csv(chr20_tsv, sep='\t')
    for tr, cols in [('MDD', ['SNP', 'A1_MDD', 'A2_MDD', 'BETA_MDD', 'SE_MDD', 'P_MDD', 'N_MDD']),
                     ('ANX', ['SNP', 'A1_MDD', 'A2_MDD', 'BETA_ANX', 'SE_ANX', 'P_ANX', 'N_ANX'])]:
        ma = d20[cols].copy()
        ma.columns = ['SNP', 'A1', 'A2', 'b', 'se', 'p', 'n']
        ma.insert(3, 'freq', 0.25)
        ma.to_csv(os.path.join(WORK_DIR, 'smr', f'chr20_{tr}.ma'), sep='\t', index=False)

if not any('chr20' in f for f in os.listdir(sqtl_cache)):
    print('[Part C] Fetching BrainMeta v2 cis-sQTL tarball (9.0 GB) -> extracting chr20 BESD only...')
    if not os.path.exists(sqtl_tar):
        subprocess.run(['wget', '-c', '--user-agent=Mozilla/5.0', sqtl_url, '-O', sqtl_tar], check=True)
    # List members matching chr20 and extract selectively to save disk space
    lst = subprocess.check_output(['tar', '-tzf', sqtl_tar], text=True).splitlines()
    chr20_members = [m for m in lst if 'chr20' in m or '20.' in m]
    if chr20_members:
        subprocess.run(['tar', '-xzf', sqtl_tar, '-C', sqtl_cache] + chr20_members, check=True)

besd_chr20 = next((os.path.join(r, f).replace('.besd', '') for r, _, fs in os.walk(sqtl_cache) for f in fs if f.endswith('.besd') and '20' in f), None)
if besd_chr20 and os.path.exists('/usr/local/bin/smr'):
    for tr in ['MDD', 'ANX']:
        out_smr = os.path.join(OUT_DIR, f'planB_BrainMeta_sQTL_chr20_{tr}')
        subprocess.run([
            '/usr/local/bin/smr',
            '--bfile', os.path.join(WORK_DIR, 'ld', 'g1000_eur'),
            '--gwas-summary', os.path.join(WORK_DIR, 'smr', f'chr20_{tr}.ma'),
            '--beqtl-summary', besd_chr20,
            '--chr', '20', '--peqtl-smr', '5e-8', '--heidi-mtd', '1',
            '--out', out_smr
        ], check=True)
    print('[SUCCESS] Part C BrainMeta sQTL SMR finished. Inspect:', OUT_DIR)
else:
    print('[Info] Place BrainMeta chr20 sQTL .besd/.epi/.esi in', sqtl_cache, 'to run SMR.')


## Part D: Independent Postmortem DLPFC snRNA-seq Replication Template (`limma-voom` + `HSPA1A` + `SVA`)

Point `EXTERNAL_H5AD` at any independent MDD cortical snRNA-seq cohort (e.g., PsychENCODE Capstone or postmortem depression replication `h5ad`), ensuring donor-level pseudobulk counts (`>= 10` nuclei per subtype per donor), chemistry/batch control, `HSPA1A` stress-proxy sensitivity, and 3-factor `sva::svaseq` testing matching our primary discovery pipeline.


In [ ]:
# [7] Part D: Donor-Level Pseudobulk Replication in an Independent MDD snRNA-seq Cohort
external_candidates = [
    os.path.join(DRIVE_ROOT, 'external_mdd_snrna.h5ad'),
    '/content/drive/MyDrive/MDD_New_Project_Pipelines_2026/MDD_PlanB_Replication/external_mdd_snrna.h5ad',
    '/content/drive/MyDrive/MDD_PlanB_Replication/external_mdd_snrna.h5ad',
    '/content/external_mdd_snrna.h5ad',
]
EXTERNAL_H5AD = next((p for p in external_candidates if os.path.exists(p)), external_candidates[0])
if os.path.exists(EXTERNAL_H5AD):
    import scanpy as sc
    adata = sc.read_h5ad(EXTERNAL_H5AD, backed=None)
    # Expected obs columns: donor_id, diagnosis (Control/MDD), batch, subtype
    obs = adata.obs[['donor_id', 'diagnosis', 'batch', 'subtype']].copy()
    genes_keep = [g for g in ['SLC12A5', 'GABRA1', 'ATP5F1D', 'HSPA1A'] if g in adata.var_names]
    print(f'[Part D] Loaded external snRNA cohort: {adata.n_obs:,} nuclei, {obs["donor_id"].nunique()} donors, markers={genes_keep}')
    # Aggregate pseudobulk per donor x subtype and export for identical R limma-voom + HSPA1A + SVA audit
    pb_rows = []
    X = adata.raw.X if adata.raw is not None else adata.X
    for (did, st_id), grp in obs.groupby(['donor_id', 'subtype'], observed=True):
        if len(grp) < 10:
            continue
        idx = obs.index.get_indexer(grp.index)
        sums = np.asarray(X[idx].sum(axis=0)).ravel()
        # Export summary count row for SLC12A5 / HSPA1A quick replication audit
        gidx = {g: list(adata.var_names).index(g) for g in genes_keep}
        row = {'donor_id': did, 'subtype': st_id, 'diagnosis': grp['diagnosis'].iloc[0], 'batch': grp['batch'].iloc[0], 'n_nuclei': len(grp), 'lib_size': float(sums.sum())}
        for g, gi in gidx.items():
            row[g] = float(sums[gi])
        pb_rows.append(row)
    pb_df = pd.DataFrame(pb_rows)
    pb_df.to_csv(os.path.join(OUT_DIR, 'planB_external_snrna_pseudobulk_slc12a5.tsv'), sep='\t', index=False)
    print('[SUCCESS] Exported donor-level pseudobulk matrix to', OUT_DIR)
else:
    print(f'[Template Ready] Stage an independent MDD snRNA-seq .h5ad at {EXTERNAL_H5AD} and re-run Cell [7].')
